# 🔬 DeepTrace v2 — EfficientNet-B4 Training Pipeline

This notebook trains an **EfficientNet-B4** model for facial deepfake detection. It implements the two critical requirements for our forensic pipeline:
1. **MTCNN Face Cropping**: Extracts faces from raw images with a 20% margin (exactly matching our FastAPI backend).
2. **Forensic Augmentations**: Simulates social media degradation (JPEG compression, blur, noise) using Albumentations to prevent real-world false negatives.

> **Reviewed & hardened.** This pass fixes three issues that would have crashed training or silently corrupted the exported model: an `HF Trainer` column-pruning bug, an unpinned Albumentations API mismatch, and an EfficientNet-B4 normalization mismatch that would have skewed every prediction the FastAPI backend makes in production. Each fix is called out inline where it happens.

### ⚡ Prerequisites:
Set your Colab Runtime to **T4 GPU** or **A100 GPU** (`Runtime > Change runtime type`).

In [ ]:
# Step 1: Install Required Dependencies
# `albumentations` is pinned to 2.0.8 -- the last release under the permissive MIT
# license before the project forked into the actively-maintained but AGPL-3.0-licensed
# "AlbumentationsX". Pinning also protects this pipeline from further breaking API
# changes (see Section 3). If this exact pin ever conflicts with another package's
# requirements, it's safe to drop the version and just `pip install albumentations` --
# every transform below already uses the current (2.x) argument names, not the
# old ones, so it will keep working either way.
# `opencv-python-headless` is specified explicitly since recent Albumentations releases
# no longer install an OpenCV backend automatically, and the headless build avoids
# libGL errors on Colab's GUI-less runtime.
!pip install -q transformers datasets accelerate torchvision evaluate kagglehub scikit-learn "albumentations==2.0.8" opencv-python-headless facenet-pytorch

## 1. Dataset Preparation & MTCNN Face Extraction
DeepTrace analyzes *faces*, not full images. We need to extract the faces from our training data before feeding them to the model.

> **Note:** Extracting faces from 100,000+ raw videos using MTCNN takes several days on a single GPU. For this template, we are downloading a high-quality dataset (`140k-real-and-fake-faces`) where the faces **have already been extracted**.
> 
> Below is the exact MTCNN extraction code we use in our backend. We include it here so you can run it if you ever want to process your own raw videos (like FaceForensics++).

In [ ]:
import os
import cv2
from PIL import Image
import numpy as np
import torch
from facenet_pytorch import MTCNN
import kagglehub

# 1. Download a pre-cropped deepfake face dataset
print("Downloading pre-cropped face dataset...")
from google.colab import userdata
import os

# Authenticate with Kaggle to get maximum download speed
# Add KAGGLE_USERNAME and KAGGLE_KEY to Colab's Secrets (the key icon on the left)
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
    print("Kaggle credentials loaded from Colab Secrets.")
    kagglehub.login()
except Exception as e:
    print("Could not authenticate. Downloading anonymously (may be slower).")

dataset_path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces") 
print(f"Dataset downloaded to: {dataset_path}")

# 2. MTCNN Reference Implementation (Matches analyzer.py)
# You only need to run this if you are importing RAW UNPROCESSED images.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(keep_all=True, min_face_size=20, thresholds=[0.5, 0.6, 0.6], device=device)

def extract_face_from_image(img_path, save_path):
    """Extracts the face with a 20% margin, matching our inference pipeline."""
    try:
        image = Image.open(img_path).convert('RGB')
        max_dim = max(image.size)
        scale_factor = 1.0
        if max_dim > 1920:
            scale_factor = 1920.0 / max_dim
            new_size = (int(image.width * scale_factor), int(image.height * scale_factor))
            detect_img = image.resize(new_size, Image.Resampling.LANCZOS)
        else:
            detect_img = image

        boxes, probs = mtcnn.detect(np.array(detect_img))
        if boxes is not None and len(boxes) > 0:
            box = boxes[0]
            prob = probs[0]
            if prob > 0.60:
                x1, y1, x2, y2 = [int(c / scale_factor) for c in box]
                w, h = x2 - x1, y2 - y1
                cx1 = max(0, int(x1 - w * 0.2))
                cy1 = max(0, int(y1 - h * 0.2))
                cx2 = min(image.width, int(x2 + w * 0.2))
                cy2 = min(image.height, int(y2 + h * 0.2))
                
                face_crop = image.crop((cx1, cy1, cx2, cy2))
                face_crop.save(save_path, quality=95)
                return True
    except Exception as e:
        pass
    return False


## 2. Load the EfficientNet-B4 Processor
We load the processor **before** building the augmentation pipeline in the next section. It carries the exact normalization statistics (`image_mean` / `image_std`) and input resolution baked into the `google/efficientnet-b4` checkpoint.

> ⚠️ **This matters more than it looks.** `google/efficientnet-b4`'s shipped `image_std` is `[0.4785, 0.4733, 0.4743]` — **not** the generic ImageNet `[0.229, 0.224, 0.225]` most tutorials (including the original version of this notebook) hardcode. Since `processor.save_pretrained()` is exactly what your FastAPI backend loads at inference time, training with one normalization and shipping a processor config with a different one would silently skew every prediction in production — no error, no warning, just quietly worse accuracy. Deriving the augmentation `Normalize` step from `processor.image_mean` / `processor.image_std` below guarantees the two always agree, even if you swap EfficientNet variants later.

In [ ]:
from transformers import AutoImageProcessor

MODEL_ID = "google/efficientnet-b4"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

IMAGE_SIZE = processor.size["height"]  # 380 for B4 -- derived, not hardcoded
NORM_MEAN = processor.image_mean
NORM_STD = processor.image_std

print(f"Resolution:     {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Normalize mean: {NORM_MEAN}")
print(f"Normalize std:  {NORM_STD}")

## 3. Aggressive Forensic Augmentations
To survive WhatsApp and Instagram compression, we *must* simulate it during training using `albumentations`.

> **Albumentations API note:** `ImageCompression`, `GaussNoise`, `ShiftScaleRotate`, `ColorJitter`, `GaussianBlur`, and `RandomResizedCrop` all renamed their constructor arguments in recent Albumentations releases -- e.g. `quality_lower`/`quality_upper` → `quality_range`; `var_limit` → `std_range` (expressed as a *fraction* of the max pixel value, not a raw variance); single values like `brightness=0.2` → explicit tuples like `brightness_range=(0.8, 1.2)`; `height=`/`width=` → `size=(h, w)`. The old names now raise a `TypeError` when `A.Compose([...])` is built, well before training starts. The cell below uses the current syntax and is pinned against `albumentations==2.0.8` (installed above).

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from datasets import load_dataset
import numpy as np

# NOTE: IMAGE_SIZE, NORM_MEAN, NORM_STD come from the processor loaded in Section 2 --
# not hardcoded -- so training-time normalization always matches what gets shipped to
# the backend via processor.save_pretrained().

# The critical augmentation stack
train_augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_range=(-0.05, 0.05),
        scale_range=(-0.05, 0.05),   # biased by 1 internally -> effective scale in (0.95, 1.05)
        rotate_range=(-15, 15),
        p=0.5,
    ),

    # Color & Lighting
    A.ColorJitter(brightness_range=(0.8, 1.2), contrast_range=(0.8, 1.2),
                  saturation_range=(0.8, 1.2), hue_range=(-0.1, 0.1), p=0.5),

    # ⚡ CRITICAL: Social Media Compression Simulation ⚡
    A.ImageCompression(quality_range=(30, 90), p=0.6),
    A.GaussianBlur(blur_range=(3, 7), p=0.3),
    A.GaussNoise(std_range=(0.0124, 0.0277), p=0.3),  # equivalent to the old var_limit=(10, 50)

    # Partial occlusion (hands, hair, masks, watermarks) -- common in real-world source
    # photos and a cheap way to stop the model leaning on any single facial region.
    A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(0.05, 0.15),
                     hole_width_range=(0.05, 0.15), p=0.15),

    # Spatial
    A.RandomResizedCrop(size=(IMAGE_SIZE, IMAGE_SIZE), scale=(0.8, 1.0)),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),  # Converts to PyTorch Tensor shape (C, H, W)
])

val_augmentations = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

def apply_train_transforms(examples):
    images = [np.array(img.convert("RGB")) for img in examples["image"]]
    examples["pixel_values"] = [train_augmentations(image=img)["image"] for img in images]
    return examples

def apply_val_transforms(examples):
    images = [np.array(img.convert("RGB")) for img in examples["image"]]
    examples["pixel_values"] = [val_augmentations(image=img)["image"] for img in images]
    return examples

print("Augmentation pipeline initialized.")

## 4. Dataset Preparation & Label Mapping
Loads the pre-cropped 140k-real-and-fake-faces dataset and enforces the exact `"Real"`/`"Fake"` label mapping the backend expects.

> **Split name note:** the dataset's raw folders are `train/`, `valid/`, `test/`. 🤗 `datasets`' `imagefolder` loader recognizes `valid` as a validation directory but exposes it under the canonical split key **`"validation"`** -- not `"valid"` -- in the resulting `DatasetDict`. Indexing with `dataset["valid"]` raises a `KeyError`. The cell below reads whichever key is actually present so it works either way.

In [ ]:
# Load Dataset into HuggingFace Datasets
print("Loading dataset...")
# Note: xhlulu/140k dataset is inside real_vs_fake/real-vs-fake/
data_dir = os.path.join(dataset_path, "real_vs_fake", "real-vs-fake")
dataset = load_dataset("imagefolder", data_dir=data_dir)
print(f"Splits found: {list(dataset.keys())}")

# datasets' imagefolder loader maps a raw "valid" folder to the split key "validation".
val_split_key = "validation" if "validation" in dataset else "valid"

# Explicitly map labels to "Real" and "Fake" for analyzer.py compatibility
class_names = dataset["train"].features["label"].names
id2label = {}
label2id = {}
for i, name in enumerate(class_names):
    # Force mapping to exactly "Real" or "Fake"
    mapped_name = "Real" if "real" in name.lower() else "Fake"
    id2label[i] = mapped_name
    label2id[mapped_name] = i

# Fail loudly rather than silently mis-mapping: if the raw folder names ever change (a
# different dataset, a re-export, etc.) and neither contains "real", every class would
# collapse onto "Fake" here -- and the backend's label2id.get("Fake", 1) lookup would
# then silently point at the wrong class instead of erroring.
assert set(id2label.values()) == {"Real", "Fake"}, (
    f"Label mapping failed -- expected exactly {{'Real', 'Fake'}}, got {set(id2label.values())}. "
    f"Raw class names were {class_names}."
)
print(f"Analyzer.py Label Mapping: {id2label}")

# Sanity-check class balance -- informs whether class weighting is needed later.
from collections import Counter
label_counts = Counter(dataset["train"]["label"])
print(f"Train class balance: { {id2label[k]: v for k, v in label_counts.items()} }")

# Apply transforms
train_ds = dataset["train"].with_transform(apply_train_transforms)
val_ds = dataset[val_split_key].with_transform(apply_val_transforms)
test_ds = dataset["test"].with_transform(apply_val_transforms)

## 5. Finish Loading the EfficientNet-B4 Model
The processor was already loaded in Section 2. We now attach the real `id2label` / `label2id` computed from the dataset and load the classification head. We use HuggingFace's `AutoModelForImageClassification` to ensure 100% compatibility with `analyzer.py`.

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print("EfficientNet-B4 loaded successfully!")
print(f"model.config.label2id = {model.config.label2id}")

## 6. Evaluation Metrics & Training Configuration
Hyperparameters and `Trainer` flags below are tuned for **full-model** fine-tuning (nothing is frozen) on a single Colab T4 (16GB).

In [ ]:
import evaluate
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels, average="weighted")["recall"],
    }

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

OUTPUT_DIR = "./deeptrace-efficientnet"

# T4 (Turing) has no bf16 tensor cores -- fp16 is correct there. A100+ (Ampere) supports
# bf16, which is more numerically stable (no loss-scaling headaches). This picks whichever
# the runtime actually has, so the notebook doesn't need editing if you move to an A100.
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",

    # Full-model fine-tuning (nothing below is frozen), so this uses a fine-tuning-
    # appropriate LR rather than the head-only LR the original comment implied. 1e-4 is
    # workable for training a fresh head on a *frozen* backbone, but applied to the whole
    # pretrained network it risks overshooting and forgetting the ImageNet features that
    # make the backbone useful in the first place.
    learning_rate=3e-5,
    weight_decay=1e-5,             # was left at the implicit default of 0.0

    per_device_train_batch_size=16,  # B4 is memory intensive
    gradient_accumulation_steps=2,   # Effective batch size = 32
    per_device_eval_batch_size=32,

    # Flip to True if you hit a CUDA OOM on the T4 -- trades ~20-30% more training time
    # for substantially lower activation memory. Try it first without; batch 16 @ 380px
    # in mixed precision usually fits in 16GB, but free-tier Colab GPU headroom varies
    # run to run.
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    num_train_epochs=5,              # Early stopping below will cut this short if it overfits
    warmup_ratio=0.1,
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,

    fp16=use_fp16,
    bf16=use_bf16,
    label_smoothing_factor=0.1,      # Prevents overconfidence

    dataloader_num_workers=2,        # Colab typically gives 2 vCPUs; raise if you have more

    # REQUIRED with .with_transform(): Trainer's default column-pruning runs BEFORE the
    # transform ever executes, so it would drop the "image" column (not a valid
    # EfficientNetForImageClassification.forward() argument) before pixel_values can be
    # built from it -- trainer.train() would crash on the very first batch without this.
    remove_unused_columns=False,

    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    # Stops training once val F1 stalls for 2 straight epochs instead of always burning
    # through all 5 -- saves Colab GPU-hours and is the main defense against overfitting
    # on top of weight decay + label smoothing + the augmentation stack above.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# 🚀 Run Training
print("Starting EfficientNet-B4 fine-tuning...")
trainer.train()

## 7. Evaluate on Test Set
Aggregate metrics plus a confusion matrix and per-class report -- for a forensic tool the two error types aren't symmetric. A **false negative** (a deepfake classified as real) is the costlier mistake, so it's worth checking that rate specifically rather than only looking at overall accuracy/F1.

In [ ]:
# 📊 Evaluate on Test Set
print("Evaluating on test set...")
metrics = trainer.evaluate(test_ds)
print(metrics)

# Per-class breakdown -- accuracy/F1 alone can hide an asymmetric error pattern.
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids
target_names = [id2label[i] for i in range(len(id2label))]

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows = true, cols = predicted):")
print(f"{'':>8}" + "".join(f"{n:>8}" for n in target_names))
for i, row in enumerate(cm):
    print(f"{target_names[i]:>8}" + "".join(f"{v:>8}" for v in row))

fake_idx = label2id["Fake"]
real_idx = label2id["Real"]
false_negatives = cm[fake_idx][real_idx]  # actual Fake, predicted Real -- the costly miss
print(f"\nFalse negatives (Fake misclassified as Real): {false_negatives} / {cm[fake_idx].sum()}")

## 8. Save, Verify & Export
Before zipping, reload the saved checkpoint fresh from disk and confirm `label2id` round-trips to exactly `{"Fake": ..., "Real": ...}` -- the same lookup your backend's `analyzer.py` performs (`self.model.config.label2id.get("Fake", 1)`). This catches a label-mapping or export mistake here, rather than after re-uploading a broken model to the backend.

In [ ]:
# 💾 Save and Export Model
import shutil
from google.colab import files

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

# Round-trip check: reload exactly like the FastAPI backend would, and confirm the label
# mapping and normalization survived the save.
_check_model = AutoModelForImageClassification.from_pretrained(OUTPUT_DIR)
_check_processor = AutoImageProcessor.from_pretrained(OUTPUT_DIR)
assert _check_model.config.label2id.get("Fake") is not None, "Saved config is missing the 'Fake' label!"
assert _check_model.config.label2id.get("Real") is not None, "Saved config is missing the 'Real' label!"
print(f"Saved label2id: {_check_model.config.label2id}")
print(f"Saved normalize mean/std: {_check_processor.image_mean} / {_check_processor.image_std}")
del _check_model, _check_processor

zip_filename = "deeptrace_efficientnet.zip"
shutil.make_archive("deeptrace_efficientnet", "zip", OUTPUT_DIR)

print(f"Created {zip_filename}. Downloading to local machine...")
files.download(zip_filename)
print("Extract this zip into backend/models/deeptrace-efficientnet and update your .env file!")